# CWR 2017–2026 encounters: all-archives report and export

Build a **separate** all-years dataset and report by combining:

- every encounter year exposed by the [CWR Archive Encounters index](https://whaleresearch.wixsite.com/archives): 2017–2023;
- the existing, independently reproducible 2024–2026 Atlist export from notebook `02_CWR_2024_2026_ECOTYPE_REPORT_AND_EXPORT.ipynb`.

The archive pages contain structured encounter fields, including source-reported start/end coordinates on many records. This notebook extracts those fields directly; it does not geocode text locations.

**Counting contract:** one output row is one canonical CWR encounter identity within a source year and record series. Separate sequence pages for the same encounter are aggregated, while every underlying page URL and checksum remains attached to the row. Standard and UAV encounter series remain distinct.

**Important:** the archive landing page says the survey began in 1976, but its encounter navigation exposes 2017–2023 only. Earlier survey years are therefore not represented as unavailable encounter archives.

In [ ]:
from __future__ import annotations

import hashlib
import html
import json
import re
import time
from collections import Counter, defaultdict
from concurrent.futures import ThreadPoolExecutor, as_completed
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

import folium
from folium.plugins import MarkerCluster
import numpy as np
import pandas as pd
import plotly.express as px
import requests
from bs4 import BeautifulSoup
from IPython.display import HTML, display


In [ ]:
ARCHIVE_INDEX_URL = "https://whaleresearch.wixsite.com/archives"
ARCHIVE_YEARS = list(range(2017, 2024))
REQUEST_TIMEOUT_SECONDS = 60
MAX_FETCH_ATTEMPTS = 3
ARCHIVE_FETCH_WORKERS = 8
REFRESH_ARCHIVE_CACHE = False

import os
NOTEBOOK_DIR = Path(os.environ["MARINE_MAMMALS_CWR_OUTPUT_ROOT"]).expanduser().resolve()

EXISTING_ATLIST_CSV = NOTEBOOK_DIR / "exports/cwr_2024_2026_atlist_encounters.csv"
CACHE_DIR = NOTEBOOK_DIR / "cache"
ARCHIVE_CACHE_PATH = CACHE_DIR / "cwr_2017_2023_archive_extracts.json"
EXPORT_DIR = NOTEBOOK_DIR / "exports"
REPORT_DIR = NOTEBOOK_DIR / "reports"
CSV_PATH = EXPORT_DIR / "cwr_2017_2026_all_encounters.csv"
REPORT_PATH = REPORT_DIR / "cwr_2017_2026_all_encounters_report.html"
REPO_ROOT = Path(os.environ["MARINE_MAMMALS_WORKSPACE_ROOT"]).expanduser().resolve()
ORCACAST_SIGHTINGS_ROOT = REPO_ROOT / "data/processed/domain/whale_layer/sightings"
ORCACAST_OBSERVATIONS_PATH = ORCACAST_SIGHTINGS_ROOT / "observations.parquet"
ORCACAST_ASSOCIATIONS_PATH = ORCACAST_SIGHTINGS_ROOT / "associations.parquet"
ORCACAST_NORMALIZE_LATEST_PATH = ORCACAST_SIGHTINGS_ROOT / "manifests/normalize/latest.json"
CWR_SOURCE_LATEST_PATH = REPO_ROOT / "data/raw/whale/sightings/cwr/latest.json"

USER_AGENT = "OrcaCast-CWR-archive-export/0.1 (bounded public research request)"

print({
    "archive_years": ARCHIVE_YEARS,
    "existing_atlist_csv": str(EXISTING_ATLIST_CSV.resolve()),
    "archive_cache": str(ARCHIVE_CACHE_PATH.resolve()),
    "combined_csv": str(CSV_PATH.resolve()),
    "report": str(REPORT_PATH.resolve()),
    "orcacast_observations": str(ORCACAST_OBSERVATIONS_PATH.resolve()),
})


## Load the existing 2024–2026 export

This notebook consumes the previously generated CSV rather than rewriting its notebook or report. JSON-valued columns are restored to lists before the archive and Atlist tables are combined.

In [ ]:
assert EXISTING_ATLIST_CSV.exists(), (
    "Run 02_CWR_2024_2026_ECOTYPE_REPORT_AND_EXPORT.ipynb before this notebook."
)

atlist = pd.read_csv(
    EXISTING_ATLIST_CSV,
    dtype={"encounter_number": "string", "source_record_id": "string", "source_record_key": "string"},
)
for column in ("pods", "individuals", "source_tags", "qc_flags"):
    atlist[column] = atlist[column].map(lambda value: json.loads(value) if pd.notna(value) and value else [])

assert len(atlist) > 0
assert set(atlist["source_year"].astype(int)) == {2024, 2025, 2026}
assert atlist["source_record_key"].is_unique

atlist_pull_times = sorted(atlist["retrieved_at_utc"].dropna().astype(str).unique().tolist())
print({"atlist_rows": len(atlist), "atlist_pull_times": atlist_pull_times})


## Fetch and cache archive source fields

The archive indexes are parsed as one record entry per paragraph. This preserves a 2020 source anomaly where two encounter entries point to the same page URL. Individual encounter pages are fetched once per unique URL with bounded concurrency and retries.

The cache stores only structured fields, URLs, response sizes, and SHA-256 checksums—not page narratives or images. Set `REFRESH_ARCHIVE_CACHE = True` to refresh the public pages.

In [ ]:
FIELD_ALIASES = {
    "date": "encounter_date",
    "encdate": "encounter_date",
    "sequence": "encounter_sequence",
    "encseq": "encounter_sequence",
    "encounternumber": "encounter_number",
    "enc": "encounter_number",
    "encstarttime": "start_time",
    "observbegin": "start_time",
    "encendtime": "end_time",
    "observend": "end_time",
    "vessel": "vessel",
    "observers": "observers",
    "staff": "staff",
    "otherobservers": "other_observers",
    "podsorecotype": "pods_or_ecotype",
    "pods": "pods_or_ecotype",
    "location": "location_description",
    "locationdescr": "location_description",
    "beginlatlong": "begin_lat_long",
    "endlatlong": "end_lat_long",
    "startlatitude": "start_latitude",
    "startlongitude": "start_longitude",
    "endlatitude": "end_latitude",
    "endlongitude": "end_longitude",
    "idsencountered": "individuals",
    "folderid": "folder_id",
}


def sha256_bytes(content: bytes) -> str:
    return hashlib.sha256(content).hexdigest()


def normalize_inline_text(value: str) -> str:
    return " ".join(value.replace(" ", " ").split())


def fetch_public_url(url: str) -> requests.Response:
    last_error: Exception | None = None
    for attempt in range(1, MAX_FETCH_ATTEMPTS + 1):
        try:
            response = requests.get(
                url,
                timeout=REQUEST_TIMEOUT_SECONDS,
                headers={"User-Agent": USER_AGENT},
            )
            response.raise_for_status()
            return response
        except Exception as exc:
            last_error = exc
            if attempt < MAX_FETCH_ATTEMPTS:
                time.sleep(0.6 * attempt)
    raise RuntimeError(f"Failed to fetch {url}: {last_error}")


def parse_archive_index(year: int, content: bytes) -> list[dict[str, Any]]:
    soup = BeautifulSoup(content, "html.parser")
    prefix = f"https://whaleresearch.wixsite.com/{year}encounters/"
    entries: list[dict[str, Any]] = []
    seen: set[tuple[str, str]] = set()

    for paragraph in soup.find_all("p"):
        hrefs: list[str] = []
        for anchor in paragraph.find_all("a", href=True):
            href = anchor["href"].split("?")[0].split("#")[0]
            if href.startswith(prefix) and href != prefix and href not in hrefs:
                hrefs.append(href)
        if not hrefs:
            continue

        index_title = normalize_inline_text(paragraph.get_text("", strip=True))
        if not re.search(r"#\s*\d+", index_title) or "•" not in index_title:
            continue
        dedupe_key = (hrefs[0], index_title)
        if dedupe_key in seen:
            continue
        seen.add(dedupe_key)

        parts = [part.strip() for part in index_title.lstrip("« ").split("•", 2)]
        number_match = re.search(r"#\s*(\d+)", parts[0])
        sequence_match = re.search(r"(?:Seq|Sequence)\s*#?\s*(\d+)", parts[0], re.I)
        if number_match is None:
            continue
        entries.append({
            "source_year": year,
            "record_series": "uav_encounter" if "uav" in hrefs[0].casefold() else "encounter",
            "encounter_number": number_match.group(1),
            "index_sequence": sequence_match.group(1) if sequence_match else None,
            "index_date_text": parts[1] if len(parts) > 1 else None,
            "index_descriptor": parts[2] if len(parts) > 2 else None,
            "index_title": index_title,
            "record_page_url": hrefs[0],
        })
    return entries


def parse_page_fields(content: bytes) -> dict[str, str]:
    soup = BeautifulSoup(content, "html.parser")
    fields: dict[str, str] = {}
    for paragraph in soup.find_all("p"):
        text = normalize_inline_text(paragraph.get_text(" ", strip=True))
        match = re.match(r"^([^:]{1,50}):\s*(.*)$", text)
        if not match:
            continue
        normalized_label = re.sub(r"[^a-z0-9]", "", match.group(1).casefold())
        canonical = FIELD_ALIASES.get(normalized_label)
        if canonical and canonical not in fields:
            fields[canonical] = match.group(2).strip()
    return fields


def fetch_page_extract(url: str) -> dict[str, Any]:
    response = fetch_public_url(url)
    return {
        "url": url,
        "status": response.status_code,
        "content_type": response.headers.get("content-type"),
        "response_bytes": len(response.content),
        "response_sha256": sha256_bytes(response.content),
        "fields": parse_page_fields(response.content),
        "error": None,
    }


In [ ]:
pipeline_archive_path = None
if CWR_SOURCE_LATEST_PATH.exists() and not REFRESH_ARCHIVE_CACHE:
    pipeline_snapshot = Path(json.loads(CWR_SOURCE_LATEST_PATH.read_text(encoding="utf-8"))["snapshot"])
    candidate = pipeline_snapshot / "archive_extracts.json"
    pipeline_archive_path = candidate if candidate.exists() else None

if pipeline_archive_path is not None:
    archive_cache = json.loads(pipeline_archive_path.read_text(encoding="utf-8"))
    assert archive_cache.get("schema_version") in {1, 2}
    print({"archive_cache_loaded": "configured_cwr_snapshot", "archive_retrieved_at_utc": archive_cache["retrieved_at_utc"]})
elif ARCHIVE_CACHE_PATH.exists() and not REFRESH_ARCHIVE_CACHE:
    archive_cache = json.loads(ARCHIVE_CACHE_PATH.read_text(encoding="utf-8"))
    assert archive_cache.get("schema_version") in {1, 2}
    print({"archive_cache_loaded": "notebook_cache", "archive_retrieved_at_utc": archive_cache["retrieved_at_utc"]})
else:
    archive_retrieved_at_utc = datetime.now(timezone.utc).isoformat(timespec="seconds").replace("+00:00", "Z")
    archive_index_results: dict[str, Any] = {}
    archive_index_entries: list[dict[str, Any]] = []

    for year in ARCHIVE_YEARS:
        index_url = f"https://whaleresearch.wixsite.com/{year}encounters"
        response = fetch_public_url(index_url)
        entries = parse_archive_index(year, response.content)
        assert entries, f"No encounter entries parsed for {year}"
        archive_index_results[str(year)] = {
            "url": index_url,
            "response_bytes": len(response.content),
            "response_sha256": sha256_bytes(response.content),
            "entry_count": len(entries),
        }
        archive_index_entries.extend(entries)

    unique_page_urls = sorted({entry["record_page_url"] for entry in archive_index_entries})
    record_pages: dict[str, Any] = {}
    with ThreadPoolExecutor(max_workers=ARCHIVE_FETCH_WORKERS) as executor:
        future_to_url = {executor.submit(fetch_page_extract, url): url for url in unique_page_urls}
        for completed, future in enumerate(as_completed(future_to_url), start=1):
            url = future_to_url[future]
            try:
                record_pages[url] = future.result()
            except Exception as exc:
                record_pages[url] = {"url": url, "status": None, "fields": {}, "error": str(exc)}
            if completed % 100 == 0 or completed == len(unique_page_urls):
                print(f"Fetched {completed:,}/{len(unique_page_urls):,} unique archive pages")

    archive_cache = {
        "schema_version": 1,
        "archive_index_url": ARCHIVE_INDEX_URL,
        "retrieved_at_utc": archive_retrieved_at_utc,
        "index_results": archive_index_results,
        "index_entries": archive_index_entries,
        "record_pages": record_pages,
    }
    CACHE_DIR.mkdir(parents=True, exist_ok=True)
    ARCHIVE_CACHE_PATH.write_text(
        json.dumps(archive_cache, ensure_ascii=False, separators=(",", ":")),
        encoding="utf-8",
    )
    print({"archive_cache_loaded": False, "archive_cache_written": str(ARCHIVE_CACHE_PATH.resolve())})

archive_index_entries = archive_cache["index_entries"]
archive_record_pages = archive_cache["record_pages"]
archive_retrieved_at_utc = archive_cache["retrieved_at_utc"]
archive_fetch_errors = [page for page in archive_record_pages.values() if page.get("error")]

print({
    "archive_index_entries": len(archive_index_entries),
    "unique_record_page_urls": len(archive_record_pages),
    "archive_fetch_errors": len(archive_fetch_errors),
    "archive_retrieved_at_utc": archive_retrieved_at_utc,
})
display(pd.DataFrame(archive_cache["index_results"]).T)


## Normalize archive encounters

Older pages use two structured schemas: 2017–2018 combine latitude/longitude in a single field, while later pages generally separate them. Coordinates are preserved as source text and converted from degrees + decimal minutes where needed. Unsigned archive longitudes are interpreted as west because the source locations are in the CWR Pacific Northwest study area; the conversion method is explicit in every row.

If an archive index identity disagrees with the linked page identity, page-level fields are not assigned to that encounter. This protects the 2020 index entry where encounters #1 and #2 share the same page URL.

In [ ]:
ECOTYPE_ORDER = [
    "Southern Resident Killer Whales",
    "Bigg's Killer Whales",
    "Northern Resident Killer Whales",
    "Unknown / not stated",
]
ECOTYPE_COLORS = {
    "Southern Resident Killer Whales": "#0072B2",
    "Bigg's Killer Whales": "#D55E00",
    "Northern Resident Killer Whales": "#009E73",
    "Unknown / not stated": "#6B7280",
}


def parse_archive_date(value: str | None, source_year: int) -> tuple[str | None, str | None, str, list[str]]:
    if not value:
        return None, None, "unknown", ["missing_archive_date"]
    raw = normalize_inline_text(value)
    normalized = re.sub(r"(?i)\bSept\b", "Sep", raw).replace(" ", "")
    qc_flags: list[str] = []
    if re.fullmatch(r"[A-Za-z]+", normalized):
        for fmt in ("%b", "%B"):
            try:
                month_number = datetime.strptime(normalized, fmt).month
                return None, f"{source_year:04d}-{month_number:02d}", "month", ["archive_month_precision_only"]
            except ValueError:
                pass
    if re.search(r"[-/]\d{3}$", normalized):
        normalized = re.sub(r"([-/])\d{3}$", rf"\g<1>{source_year}", normalized)
        qc_flags.append("archive_date_year_repaired_from_source_year")
    for fmt in (
        "%d-%b-%y", "%d-%B-%y", "%d/%m/%y", "%d-%m-%y",
        "%d-%b-%Y", "%d-%B-%Y", "%d/%m/%Y", "%d-%m-%Y",
        "%d-%b", "%d-%B",
    ):
        try:
            parsed = datetime.strptime(normalized, fmt).date()
            if "%Y" not in fmt and "%y" not in fmt:
                parsed = parsed.replace(year=source_year)
            if parsed.year != source_year:
                qc_flags.append("archive_date_year_mismatch")
            return parsed.isoformat(), parsed.strftime("%Y-%m"), "day", qc_flags
        except ValueError:
            pass
    return None, None, "unknown", ["unparsed_archive_date"]


def normalize_time(value: str | None) -> str | None:
    if not value:
        return None
    compact = re.sub(r"\s+", " ", value.strip()).upper()
    for fmt in ("%I:%M %p", "%I %p", "%H:%M", "%H%M"):
        try:
            return datetime.strptime(compact, fmt).strftime("%H:%M")
        except ValueError:
            pass
    return None


def classify_ecotype(value: str | None) -> str | None:
    text = (value or "").casefold()
    if "northern resident" in text:
        return "Northern Resident Killer Whales"
    if (
        "southern resident" in text
        or "srkw" in text
        or "jpod" in text
        or re.match(r"\s*[jkl](?:\s*[,/&]|\s+and\s+|\s+pod)", text)
    ):
        return "Southern Resident Killer Whales"
    if "transient" in text or "bigg" in text:
        return "Bigg's Killer Whales"
    return None


def parse_resident_pods(value: str | None, ecotype: str | None) -> list[str]:
    if ecotype != "Southern Resident Killer Whales":
        return []
    text = (value or "").upper().replace("JPOD", "J POD")
    return [pod for pod in ("J", "K", "L") if re.search(rf"(?:^|[^A-Z]){pod}(?:[^A-Z]|$)", text)]


def parse_coordinate_component(value: str | None, axis: str) -> tuple[float | None, str | None]:
    if not value:
        return None, None
    text = normalize_inline_text(value).upper().replace("°", " ").replace("′", " ").replace("'", " ")
    hemisphere = next((token for token in ("N", "S", "E", "W") if re.search(rf"{token}", text)), None)
    numbers = re.findall(r"-?\d+(?:\.\d+)?", text)
    if not numbers:
        return None, None
    first = float(numbers[0])
    if len(numbers) >= 2:
        value_dd = abs(first) + float(numbers[1]) / 60.0
        method = "degrees_decimal_minutes"
    else:
        value_dd = abs(first)
        method = "decimal_degrees"
    negative = first < 0 or hemisphere in {"S", "W"}
    if axis == "longitude" and first >= 0 and hemisphere not in {"E", "W"}:
        negative = True
        method += "_west_inferred_from_archive_domain"
    if negative:
        value_dd = -value_dd
    limit = 90 if axis == "latitude" else 180
    if not -limit <= value_dd <= limit:
        return None, "invalid_coordinate"
    return value_dd, method


def page_coordinates(fields: dict[str, str]) -> dict[str, Any]:
    start_lat_text = fields.get("start_latitude")
    start_lon_text = fields.get("start_longitude")
    end_lat_text = fields.get("end_latitude")
    end_lon_text = fields.get("end_longitude")
    if (not start_lat_text or not start_lon_text) and fields.get("begin_lat_long") and "/" in fields["begin_lat_long"]:
        start_lat_text, start_lon_text = [part.strip() for part in fields["begin_lat_long"].split("/", 1)]
    if (not end_lat_text or not end_lon_text) and fields.get("end_lat_long") and "/" in fields["end_lat_long"]:
        end_lat_text, end_lon_text = [part.strip() for part in fields["end_lat_long"].split("/", 1)]

    start_lat, start_lat_method = parse_coordinate_component(start_lat_text, "latitude")
    start_lon, start_lon_method = parse_coordinate_component(start_lon_text, "longitude")
    end_lat, end_lat_method = parse_coordinate_component(end_lat_text, "latitude")
    end_lon, end_lon_method = parse_coordinate_component(end_lon_text, "longitude")
    parsed_candidates = {
        "start_lat": start_lat,
        "start_lon": start_lon,
        "end_lat": end_lat,
        "end_lon": end_lon,
    }
    coordinate_qc_flags: list[str] = []
    if start_lat is not None and start_lon is not None and not (45 <= start_lat <= 52 and -130 <= start_lon <= -120):
        coordinate_qc_flags.append("archive_start_coordinate_outside_domain")
        start_lat = start_lon = None
    if end_lat is not None and end_lon is not None and not (45 <= end_lat <= 52 and -130 <= end_lon <= -120):
        coordinate_qc_flags.append("archive_end_coordinate_outside_domain")
        end_lat = end_lon = None
    methods = sorted({method for method in (start_lat_method, start_lon_method, end_lat_method, end_lon_method) if method})
    return {
        "start_lat": start_lat,
        "start_lon": start_lon,
        "end_lat": end_lat,
        "end_lon": end_lon,
        "coordinate_parse_method": ";".join(methods) if methods else None,
        "coordinate_qc_flags": coordinate_qc_flags,
        "coordinate_source_text": {
            "start_latitude": start_lat_text,
            "start_longitude": start_lon_text,
            "end_latitude": end_lat_text,
            "end_longitude": end_lon_text,
            "parsed_candidate": parsed_candidates,
        },
    }


In [ ]:
archive_sequence_rows: list[dict[str, Any]] = []

for entry in archive_index_entries:
    page = archive_record_pages[entry["record_page_url"]]
    fields = page.get("fields") or {}
    qc_flags: list[str] = []
    if page.get("error"):
        qc_flags.append("archive_page_fetch_failed")

    page_number = re.search(r"\d+", fields.get("encounter_number", ""))
    page_number_value = str(int(page_number.group(0))) if page_number else None
    identity_matches = page_number_value in {None, entry["encounter_number"]}
    if not identity_matches:
        qc_flags.append("archive_index_page_identity_mismatch")
        usable_fields: dict[str, str] = {}
    else:
        usable_fields = fields

    page_date, page_month, page_precision, page_date_flags = parse_archive_date(
        usable_fields.get("encounter_date"), entry["source_year"]
    )
    index_date, index_month, index_precision, index_date_flags = parse_archive_date(
        entry.get("index_date_text"), entry["source_year"]
    )
    page_index_date_mismatch = bool(page_date and index_date and page_date != index_date)
    if page_index_date_mismatch:
        qc_flags.append("archive_index_page_date_mismatch")
        encounter_date = index_date
        encounter_month = index_month
        date_precision = index_precision
        chosen_date_source_text = entry.get("index_date_text")
        date_parse_method = "archive_index_title_on_source_disagreement"
    else:
        encounter_date = page_date or index_date
        encounter_month = page_month or index_month
        date_precision = page_precision if page_month else index_precision
        chosen_date_source_text = usable_fields.get("encounter_date") or entry.get("index_date_text")
        date_parse_method = "archive_page" if page_date or page_month else "archive_index_title"
    qc_flags.extend(page_date_flags if usable_fields.get("encounter_date") else index_date_flags)

    page_ecotype_text = usable_fields.get("pods_or_ecotype")
    index_ecotype_text = entry.get("index_descriptor")
    ecotype_source_values = [
        value for value in dict.fromkeys((page_ecotype_text, index_ecotype_text)) if value
    ]
    ecotype_source_text = " | ".join(ecotype_source_values) if ecotype_source_values else None
    ecotype = classify_ecotype(page_ecotype_text) or classify_ecotype(index_ecotype_text)
    if ecotype is None:
        qc_flags.append("unmapped_archive_ecotype")
    coordinates = page_coordinates(usable_fields)
    qc_flags.extend(coordinates["coordinate_qc_flags"])

    sequence_text = usable_fields.get("encounter_sequence") or entry.get("index_sequence")
    sequence_number_match = re.search(r"\d+", sequence_text or "")
    sequence_sort = int(sequence_number_match.group(0)) if sequence_number_match else 9999
    archive_sequence_rows.append({
        **entry,
        "page_fields": fields,
        "usable_fields": usable_fields,
        "page_response_sha256": page.get("response_sha256"),
        "page_response_bytes": page.get("response_bytes"),
        "encounter_date": encounter_date,
        "encounter_month": encounter_month,
        "date_precision": date_precision,
        "chosen_date_source_text": chosen_date_source_text,
        "date_parse_method": date_parse_method,
        "ecotype": ecotype,
        "ecotype_source_text": ecotype_source_text,
        "sequence_text": sequence_text,
        "sequence_sort": sequence_sort,
        "qc_flags": sorted(set(qc_flags)),
        **{key: value for key, value in coordinates.items() if key != "coordinate_qc_flags"},
    })

archive_groups: dict[tuple[int, str, str], list[dict[str, Any]]] = defaultdict(list)
for row in archive_sequence_rows:
    archive_groups[(row["source_year"], row["record_series"], row["encounter_number"])].append(row)

archive_records: list[dict[str, Any]] = []
for (source_year, record_series, encounter_number), pages in sorted(archive_groups.items()):
    pages = sorted(pages, key=lambda row: (row["sequence_sort"], row["record_page_url"]))
    qc_flags = sorted({flag for page in pages for flag in page["qc_flags"]})
    if len(pages) > 1:
        qc_flags.append("multiple_sequence_pages_aggregated")

    dates = [value for value in dict.fromkeys(page["encounter_date"] for page in pages) if value]
    months = [value for value in dict.fromkeys(page["encounter_month"] for page in pages) if value]
    ecotypes = [value for value in dict.fromkeys(page["ecotype"] for page in pages) if value]
    if len(dates) > 1:
        qc_flags.append("sequence_date_disagreement")
    if len(ecotypes) > 1:
        qc_flags.append("sequence_ecotype_disagreement")

    first_start = next(
        (page for page in pages if page["start_lat"] is not None and page["start_lon"] is not None), None
    )
    last_end = next(
        (page for page in reversed(pages) if page["end_lat"] is not None and page["end_lon"] is not None), None
    )
    if first_start:
        map_lat, map_lon = first_start["start_lat"], first_start["start_lon"]
        coordinate_role = "archive_first_sequence_start"
        coordinate_status = "source_start_coordinate"
    elif last_end:
        map_lat, map_lon = last_end["end_lat"], last_end["end_lon"]
        coordinate_role = "archive_last_sequence_end_fallback"
        coordinate_status = "source_end_coordinate_fallback"
    else:
        map_lat = map_lon = None
        coordinate_role = None
        coordinate_status = "source_coordinate_unavailable"

    locations = [
        value for value in dict.fromkeys(page["usable_fields"].get("location_description") for page in pages)
        if value
    ]
    source_record_urls = [page["record_page_url"] for page in pages]
    source_record_names = [page["index_title"] for page in pages]
    page_checksums = [page["page_response_sha256"] for page in pages if page["page_response_sha256"]]
    source_payload_sha256 = hashlib.sha256("|".join(page_checksums).encode("utf-8")).hexdigest()
    ecotype = ecotypes[0] if ecotypes else None
    ecotype_source_texts = [
        value for value in dict.fromkeys(page["ecotype_source_text"] for page in pages) if value
    ]
    page_methods = [
        value for value in dict.fromkeys(page["coordinate_parse_method"] for page in pages) if value
    ]
    archive_index_info = archive_cache["index_results"][str(source_year)]
    source_record_id = f"{source_year}:{record_series}:{encounter_number}"

    archive_records.append({
        "source": "center_for_whale_research",
        "source_system": "wix_archive",
        "source_year": source_year,
        "source_map_id": None,
        "source_record_id": source_record_id,
        "source_record_key": f"wix_archive:{source_record_id}",
        "source_record_name": source_record_names[0],
        "record_type": "uav_encounter_archive" if record_series == "uav_encounter" else "encounter_archive",
        "record_series": record_series,
        "encounter_id": None,
        "encounter_number": encounter_number,
        "date": dates[0] if dates else None,
        "month": months[0] if months else None,
        "date_source_text": pages[0]["chosen_date_source_text"],
        "date_parse_method": pages[0]["date_parse_method"],
        "date_precision": pages[0]["date_precision"],
        "start_time": normalize_time(pages[0]["usable_fields"].get("start_time")),
        "start_time_source_text": pages[0]["usable_fields"].get("start_time"),
        "end_time": normalize_time(pages[-1]["usable_fields"].get("end_time")),
        "end_time_source_text": pages[-1]["usable_fields"].get("end_time"),
        "time_zone": None,
        "ecotype": ecotype,
        "ecotype_source_text": " | ".join(ecotype_source_texts) if ecotype_source_texts else None,
        "pods": parse_resident_pods(" | ".join(ecotype_source_texts), ecotype),
        "individuals": [],
        "location_description": " → ".join(locations) if locations else None,
        "map_lat": map_lat,
        "map_lon": map_lon,
        "coordinate_crs": "EPSG:4326" if map_lat is not None and map_lon is not None else None,
        "coordinate_role": coordinate_role,
        "coordinate_status": coordinate_status,
        "coordinate_parse_method": ";".join(page_methods) if page_methods else None,
        "start_lat": first_start["start_lat"] if first_start else None,
        "start_lon": first_start["start_lon"] if first_start else None,
        "end_lat": last_end["end_lat"] if last_end else None,
        "end_lon": last_end["end_lon"] if last_end else None,
        "summary": None,
        "source_tags": ecotype_source_texts,
        "source_url": source_record_urls[0],
        "source_map_url": None,
        "source_endpoint_url": None,
        "source_record_created_at": None,
        "source_record_updated_at": None,
        "source_map_created_at": None,
        "source_map_updated_at": None,
        "retrieved_at_utc": archive_retrieved_at_utc,
        "source_payload_sha256": source_payload_sha256,
        "source_record_urls": source_record_urls,
        "source_record_names": source_record_names,
        "source_record_page_count": len(pages),
        "source_record_page_sha256": page_checksums,
        "source_archive_index_url": archive_index_info["url"],
        "source_archive_index_sha256": archive_index_info["response_sha256"],
        "archive_source_fields": [page["page_fields"] for page in pages],
        "archive_coordinate_source_text": [page["coordinate_source_text"] for page in pages],
        "qc_flags": sorted(set(qc_flags)),
    })

archive = pd.DataFrame(archive_records)
assert archive["source_record_key"].is_unique
assert set(archive["source_year"].astype(int)) == set(ARCHIVE_YEARS)
assert archive["month"].notna().all()
assert archive["map_lat"].dropna().between(-90, 90).all()
assert archive["map_lon"].dropna().between(-180, 180).all()

print({
    "archive_index_entries": len(archive_index_entries),
    "archive_unique_encounters": len(archive),
    "multi_sequence_encounters": int(archive["source_record_page_count"].gt(1).sum()),
    "archive_mapped_encounters": int(archive[["map_lat", "map_lon"]].notna().all(axis=1).sum()),
})
display(archive.groupby(["source_year", "ecotype"], dropna=False).size().unstack(fill_value=0))


## Combine archive and Atlist records

The combined export retains a common source/provenance schema. Fields unavailable in one source system remain null. Archive-only arrays are JSON-serialized in the CSV alongside the existing array-valued fields.

In [ ]:
atlist = atlist.copy()
atlist["record_series"] = "encounter"
atlist["month"] = pd.to_datetime(atlist["date"], errors="coerce").dt.strftime("%Y-%m")
atlist["date_precision"] = "day"
atlist["ecotype_source_text"] = atlist["ecotype"]
atlist["coordinate_status"] = atlist.apply(
    lambda row: "atlist_map_marker" if pd.notna(row["map_lat"]) and pd.notna(row["map_lon"]) else "source_coordinate_unavailable",
    axis=1,
)
atlist["coordinate_parse_method"] = "atlist_signed_decimal_degrees"
atlist["source_record_urls"] = [[] for _ in range(len(atlist))]
atlist["source_record_names"] = atlist["source_record_name"].map(lambda value: [value] if pd.notna(value) else [])
atlist["source_record_page_count"] = 1
atlist["source_record_page_sha256"] = atlist["source_payload_sha256"].map(lambda value: [value] if pd.notna(value) else [])
atlist["source_archive_index_url"] = None
atlist["source_archive_index_sha256"] = None
atlist["archive_source_fields"] = [[] for _ in range(len(atlist))]
atlist["archive_coordinate_source_text"] = [[] for _ in range(len(atlist))]

COMBINED_COLUMNS = [
    "source", "source_system", "source_year", "source_map_id", "source_record_id",
    "source_record_key", "source_record_name", "record_type", "record_series",
    "encounter_id", "encounter_number", "date", "month", "date_source_text",
    "date_parse_method", "date_precision",
    "start_time", "start_time_source_text", "end_time", "end_time_source_text", "time_zone",
    "ecotype", "ecotype_source_text", "pods", "individuals", "location_description",
    "map_lat", "map_lon", "coordinate_crs", "coordinate_role", "coordinate_status",
    "coordinate_parse_method", "start_lat", "start_lon", "end_lat", "end_lon",
    "summary", "source_tags", "source_url", "source_map_url", "source_endpoint_url",
    "source_record_created_at", "source_record_updated_at",
    "source_map_created_at", "source_map_updated_at", "retrieved_at_utc",
    "source_payload_sha256", "source_record_urls", "source_record_names",
    "source_record_page_count", "source_record_page_sha256",
    "source_archive_index_url", "source_archive_index_sha256",
    "archive_source_fields", "archive_coordinate_source_text", "qc_flags",
]

for frame in (archive, atlist):
    for column in COMBINED_COLUMNS:
        if column not in frame.columns:
            frame[column] = None

encounters = pd.concat([archive[COMBINED_COLUMNS], atlist[COMBINED_COLUMNS]], ignore_index=True)
encounters["source_year"] = encounters["source_year"].astype(int)
encounters["ecotype_display"] = encounters["ecotype"].fillna("Unknown / not stated")
encounters["month_dt"] = pd.to_datetime(encounters["month"] + "-01", errors="coerce")
encounters = encounters.sort_values(
    ["month_dt", "source_year", "record_series", "encounter_number", "source_record_key"],
    na_position="last",
).reset_index(drop=True)

assert len(encounters) == len(archive) + len(atlist)
assert encounters["source_record_key"].is_unique
assert set(encounters["source_year"]) == set(range(2017, 2027))
assert encounters["month_dt"].notna().all()
assert encounters["map_lat"].dropna().between(-90, 90).all()
assert encounters["map_lon"].dropna().between(-180, 180).all()

print({
    "combined_encounters": len(encounters),
    "archive_encounters": len(archive),
    "atlist_encounters": len(atlist),
    "mapped_encounters": int(encounters[["map_lat", "map_lon"]].notna().all(axis=1).sum()),
    "source_years": [int(encounters["source_year"].min()), int(encounters["source_year"].max())],
})


## Metrics and time series

The monthly series includes all canonical archive and Atlist encounters. Zeroes are filled only from January 2017 through the latest in-year 2026 encounter month in the existing Atlist export; later 2026 months are not shown as zero.

In [ ]:
ecotype_counts = (
    encounters.groupby("ecotype_display", as_index=False, observed=True)
    .agg(encounters=("source_record_key", "count"))
    .sort_values("encounters", ascending=False)
)
year_ecotype_counts = (
    encounters.pivot_table(
        index="source_year",
        columns="ecotype_display",
        values="source_record_key",
        aggfunc="count",
        fill_value=0,
    )
    .reindex(columns=[column for column in ECOTYPE_ORDER if column in encounters["ecotype_display"].unique()], fill_value=0)
    .astype(int)
)
year_ecotype_counts["Total"] = year_ecotype_counts.sum(axis=1)
year_ecotype_counts.columns.name = None

latest_in_year_month = encounters.loc[
    encounters["month_dt"].dt.year.eq(encounters["source_year"]), "month_dt"
].max()
series_months = pd.date_range("2017-01-01", latest_in_year_month, freq="MS")
series_ecotypes = [value for value in ECOTYPE_ORDER if value in encounters["ecotype_display"].unique()]
series_grid = pd.MultiIndex.from_product(
    [series_months, series_ecotypes], names=["month_dt", "ecotype_display"]
).to_frame(index=False)
monthly_observed = (
    encounters.groupby(["month_dt", "ecotype_display"], as_index=False, observed=True)
    .agg(encounters=("source_record_key", "count"))
)
monthly_counts = series_grid.merge(monthly_observed, how="left", on=["month_dt", "ecotype_display"])
monthly_counts["encounters"] = monthly_counts["encounters"].fillna(0).astype(int)

display(ecotype_counts)
display(year_ecotype_counts)

time_series_fig = px.line(
    monthly_counts,
    x="month_dt",
    y="encounters",
    color="ecotype_display",
    color_discrete_map=ECOTYPE_COLORS,
    category_orders={"ecotype_display": ECOTYPE_ORDER},
    markers=False,
    labels={"month_dt": "Encounter month", "encounters": "Canonical encounters", "ecotype_display": "Ecotype"},
    title="Monthly CWR encounters by ecotype, 2017–2026",
)
time_series_fig.update_layout(
    template="plotly_white",
    hovermode="x unified",
    legend_title_text="Ecotype",
    margin=dict(l=50, r=30, t=75, b=50),
)
time_series_fig.update_yaxes(rangemode="tozero", dtick=5)
time_series_fig.show()


## Encounter map

Each mapped archive encounter uses its first available source-reported sequence start coordinate; if only an end coordinate exists, the map uses that explicit fallback. Atlist rows retain their original marker coordinates. Layer controls toggle source years, and popups link to the encounter source where available.

In [ ]:
mapped = encounters.dropna(subset=["map_lat", "map_lon"]).copy()
center = [mapped["map_lat"].mean(), mapped["map_lon"].mean()]

encounter_map = folium.Map(location=center, zoom_start=6, tiles="CartoDB positron", control_scale=True)
for source_year, group in mapped.groupby("source_year", sort=True):
    year_layer = folium.FeatureGroup(name=f"{source_year} ({len(group):,} mapped)", show=True)
    cluster = MarkerCluster(name=f"{source_year} encounters", control=False).add_to(year_layer)
    for row in group.itertuples(index=False):
        ecotype = row.ecotype_display
        color = ECOTYPE_COLORS.get(ecotype, ECOTYPE_COLORS["Unknown / not stated"])
        location = row.location_description if pd.notna(row.location_description) else "Not stated"
        source_link = (
            f'<a href="{html.escape(str(row.source_url), quote=True)}" target="_blank" rel="noreferrer">Open source record</a>'
            if pd.notna(row.source_url) else "Source record link unavailable"
        )
        popup_html = (
            f"<strong>{row.source_year} {html.escape(str(row.record_series))} #{html.escape(str(row.encounter_number))}</strong><br>"
            f"Date: {html.escape(str(row.date if pd.notna(row.date) else row.month))}<br>"
            f"Ecotype: {html.escape(ecotype)}<br>"
            f"Location: {html.escape(str(location))}<br>"
            f"Coordinate role: {html.escape(str(row.coordinate_role))}<br>"
            f"Source: {html.escape(str(row.source_system))}<br>{source_link}"
        )
        folium.CircleMarker(
            location=[row.map_lat, row.map_lon],
            radius=4.6,
            color="#FFFFFF",
            weight=1.0,
            fill=True,
            fill_color=color,
            fill_opacity=0.84,
            tooltip=f"{row.source_year} encounter #{row.encounter_number} · {row.date if pd.notna(row.date) else row.month}",
            popup=folium.Popup(popup_html, max_width=420),
        ).add_to(cluster)
    year_layer.add_to(encounter_map)

folium.LayerControl(collapsed=True).add_to(encounter_map)
legend_items = "".join(
    f'<div style="margin:4px 0"><span style="display:inline-block;width:11px;height:11px;border-radius:50%;background:{color};margin-right:7px"></span>{html.escape(label)}</div>'
    for label, color in ECOTYPE_COLORS.items()
    if label in mapped["ecotype_display"].unique()
)
legend = f'''
<div style="position:fixed;bottom:24px;left:24px;z-index:9999;background:white;padding:10px 12px;border:1px solid #CBD5E1;border-radius:6px;box-shadow:0 2px 8px rgba(0,0,0,.15);font:12px/1.3 sans-serif">
<strong>Ecotype</strong>{legend_items}
</div>
'''
encounter_map.get_root().html.add_child(folium.Element(legend))
encounter_map


## All-years CSV export

Array and object fields are serialized as compact JSON strings. Blank scalar fields mean unavailable or unstated—not zero.

In [ ]:
JSON_COLUMNS = [
    "pods", "individuals", "source_tags", "source_record_urls", "source_record_names",
    "source_record_page_sha256", "archive_source_fields", "archive_coordinate_source_text", "qc_flags",
]

export_df = encounters[COMBINED_COLUMNS].copy()
for column in JSON_COLUMNS:
    export_df[column] = export_df[column].map(
        lambda value: json.dumps(value if isinstance(value, (list, dict)) else [], ensure_ascii=False, separators=(",", ":"))
    )

EXPORT_DIR.mkdir(parents=True, exist_ok=True)
export_df.to_csv(CSV_PATH, index=False, lineterminator="\n")
csv_sha256 = hashlib.sha256(CSV_PATH.read_bytes()).hexdigest()

reloaded = pd.read_csv(
    CSV_PATH,
    dtype={"encounter_number": "string", "source_record_id": "string", "source_record_key": "string"},
)
assert len(reloaded) == len(export_df)
assert reloaded["source_record_key"].nunique(dropna=True) == len(export_df)
assert pd.to_numeric(reloaded["map_lat"], errors="coerce").dropna().between(-90, 90).all()
assert pd.to_numeric(reloaded["map_lon"], errors="coerce").dropna().between(-180, 180).all()
for column in JSON_COLUMNS:
    reloaded[column].map(json.loads)

print({
    "csv_path": str(CSV_PATH.resolve()),
    "rows": len(reloaded),
    "columns": len(reloaded.columns),
    "csv_bytes": CSV_PATH.stat().st_size,
    "csv_sha256": csv_sha256,
})


## Comparison with the existing canonical sightings dataset

This is an auditable overlap screen against the current normalized sightings observations, not a replacement for pipeline identity resolution. It separates exact shared whale/social-group evidence from same-day spatial proximity. Broad pod or ecotype labels alone do not establish identity.

In [ ]:
COMPARISON_ECOTYPE_MAP = {
    "Bigg's Killer Whales": "TRANSIENT",
    "Southern Resident Killer Whales": "SRKW",
    "Northern Resident Killer Whales": "NRKW",
}
COMPARISON_STATUS_ORDER = [
    "Identity-supported overlap candidate",
    "Very close same-day candidate",
    "Nearby same-day candidate",
    "No close existing record",
    "No same-day existing record",
    "No ecotype-compatible same-day record",
    "Not spatially assessable",
]


def comparison_haversine_km(lat1: float, lon1: float, lat2: np.ndarray, lon2: np.ndarray) -> np.ndarray:
    radius_km = 6371.0088
    phi1 = np.radians(lat1)
    phi2 = np.radians(lat2)
    dphi = np.radians(lat2 - lat1)
    dlambda = np.radians(lon2 - lon1)
    a = np.sin(dphi / 2.0) ** 2 + np.cos(phi1) * np.cos(phi2) * np.sin(dlambda / 2.0) ** 2
    return 2.0 * radius_km * np.arctan2(np.sqrt(a), np.sqrt(1.0 - a))


def normalize_comparison_identifier(value: Any) -> str | None:
    text = re.sub(r"[^A-Z0-9]", "", str(value).upper())
    if not text:
        return None
    match = re.fullmatch(r"([TJKL])0*(\d+)([A-Z0-9]*)", text)
    if match:
        return f"{match.group(1)}{int(match.group(2))}{match.group(3)}"
    return text


normalize_latest = json.loads(ORCACAST_NORMALIZE_LATEST_PATH.read_text(encoding="utf-8"))
normalize_manifest_path = Path(normalize_latest["manifest"])
normalize_manifest = json.loads(normalize_manifest_path.read_text(encoding="utf-8"))
comparison_snapshot_id = normalize_manifest["data_snapshot"]["snapshot_id"]
comparison_snapshot_created_at = normalize_manifest["created_at"]
comparison_snapshot_sources = [
    item["source"].upper() for item in normalize_manifest["data_snapshot"]["source_watermarks"]
    if item["source"].upper() != "CWR"
]

comparison_observation_columns = [
    "OBSERVATION_ID", "EVENT_DATE", "LATITUDE", "LONGITUDE", "ECOTYPE_DETAIL",
    "SOURCE", "SOURCE_RECORD_ID", "SOURCE_RECORD_IDS",
]
orcacast_observations_all = pd.read_parquet(ORCACAST_OBSERVATIONS_PATH, columns=comparison_observation_columns)
cwr_reference_rows_excluded = int(orcacast_observations_all["SOURCE_RECORD_IDS"].map(
    lambda values: any(str(value).upper().startswith("CWR:") for value in values)
).sum())
orcacast_observations_all = orcacast_observations_all.loc[
    ~orcacast_observations_all["SOURCE_RECORD_IDS"].map(
        lambda values: any(str(value).upper().startswith("CWR:") for value in values)
    )
].copy()
orcacast_observations = orcacast_observations_all.loc[
    pd.to_datetime(orcacast_observations_all["EVENT_DATE"]).dt.year.between(2017, 2026)
].copy()
orcacast_observations["EVENT_DATE"] = pd.to_datetime(orcacast_observations["EVENT_DATE"]).dt.date

comparison_associations = pd.read_parquet(
    ORCACAST_ASSOCIATIONS_PATH,
    columns=["OBSERVATION_ID", "ASSOCIATION_KIND", "ASSOCIATION_VALUE"],
)
comparison_associations = comparison_associations.loc[
    comparison_associations["OBSERVATION_ID"].isin(orcacast_observations["OBSERVATION_ID"])
    & comparison_associations["ASSOCIATION_KIND"].isin(["MEMBER", "SOCIAL_GROUP"])
].copy()
comparison_associations["normalized_id"] = comparison_associations["ASSOCIATION_VALUE"].map(
    normalize_comparison_identifier
)
comparison_association_sets = (
    comparison_associations.dropna(subset=["normalized_id"])
    .groupby("OBSERVATION_ID")["normalized_id"]
    .agg(lambda values: set(values))
    .to_dict()
)
observations_by_date = {
    event_date: frame.copy() for event_date, frame in orcacast_observations.groupby("EVENT_DATE", sort=False)
}

comparison_rows = []
for row in encounters.itertuples(index=False):
    event_date = pd.to_datetime(row.date).date()
    desired_ecotype = COMPARISON_ECOTYPE_MAP.get(row.ecotype, "UNKNOWN")
    cwr_ids = {
        token for value in (row.individuals if isinstance(row.individuals, list) else [])
        if (token := normalize_comparison_identifier(value))
    }
    result = {
        "source_record_key": row.source_record_key,
        "source_year": int(row.source_year),
        "source_system": row.source_system,
        "ecotype": row.ecotype_display,
        "comparison_status": None,
        "comparison_basis": None,
        "existing_observation_id": None,
        "existing_preferred_source": None,
        "existing_contributing_sources": [],
        "distance_km": None,
        "shared_identifiers": [],
    }
    same_day = observations_by_date.get(event_date)
    if same_day is None or same_day.empty:
        result["comparison_status"] = "No same-day existing record"
        result["comparison_basis"] = "No canonical existing observation on the CWR calendar date"
        comparison_rows.append(result)
        continue

    compatible = same_day.copy() if desired_ecotype == "UNKNOWN" else same_day.loc[
        same_day["ECOTYPE_DETAIL"].isin([desired_ecotype, "UNKNOWN", "MIXED"])
    ].copy()
    if compatible.empty:
        result["comparison_status"] = "No ecotype-compatible same-day record"
        result["comparison_basis"] = "Existing records occur that day, but all have conflicting known ecotypes"
        comparison_rows.append(result)
        continue

    shared_by_observation = {}
    if cwr_ids:
        for observation_id in compatible["OBSERVATION_ID"]:
            shared = sorted(cwr_ids & comparison_association_sets.get(observation_id, set()))
            if shared:
                shared_by_observation[observation_id] = shared

    if pd.notna(row.map_lat) and pd.notna(row.map_lon):
        compatible["_distance_km"] = comparison_haversine_km(
            float(row.map_lat), float(row.map_lon),
            compatible["LATITUDE"].to_numpy(float), compatible["LONGITUDE"].to_numpy(float),
        )

    chosen = None
    if shared_by_observation:
        identity_candidates = compatible.loc[compatible["OBSERVATION_ID"].isin(shared_by_observation)].copy()
        if "_distance_km" in identity_candidates:
            identity_candidates = identity_candidates.sort_values("_distance_km")
        chosen = identity_candidates.iloc[0]
        result["comparison_status"] = "Identity-supported overlap candidate"
        result["comparison_basis"] = "Same date and exact shared whale or Bigg's social-group identifier"
        result["shared_identifiers"] = shared_by_observation[chosen["OBSERVATION_ID"]]
    elif "_distance_km" in compatible:
        chosen = compatible.sort_values("_distance_km").iloc[0]
        distance_km = float(chosen["_distance_km"])
        if distance_km <= 0.5:
            result["comparison_status"] = "Very close same-day candidate"
            result["comparison_basis"] = "Same date, non-conflicting ecotype, and within 500 m"
        elif distance_km <= 5.0:
            result["comparison_status"] = "Nearby same-day candidate"
            result["comparison_basis"] = "Same date, non-conflicting ecotype, and 0.5–5 km apart"
        else:
            result["comparison_status"] = "No close existing record"
            result["comparison_basis"] = "Nearest same-day ecotype-compatible observation is more than 5 km away"
    else:
        result["comparison_status"] = "Not spatially assessable"
        result["comparison_basis"] = "CWR record has no map-ready coordinate and no exact shared identifier"

    if chosen is not None:
        result["existing_observation_id"] = chosen["OBSERVATION_ID"]
        result["existing_preferred_source"] = chosen["SOURCE"]
        source_ids = chosen["SOURCE_RECORD_IDS"]
        source_ids = list(source_ids) if isinstance(source_ids, (list, tuple, np.ndarray)) else []
        result["existing_contributing_sources"] = sorted({
            str(source_id).split(":", 1)[0] for source_id in source_ids
        })
        if "_distance_km" in chosen.index and pd.notna(chosen["_distance_km"]):
            result["distance_km"] = round(float(chosen["_distance_km"]), 3)
    comparison_rows.append(result)

comparison = pd.DataFrame(comparison_rows)
comparison["comparison_status"] = pd.Categorical(
    comparison["comparison_status"], categories=COMPARISON_STATUS_ORDER, ordered=True
)
comparison_status_report = (
    comparison.groupby("comparison_status", observed=False)
    .size().rename("CWR encounters").reset_index()
)
comparison_status_report["Share"] = comparison_status_report["CWR encounters"].map(
    lambda value: f"{100 * value / len(comparison):.1f}%"
)
comparison_year_report = pd.crosstab(comparison["source_year"], comparison["comparison_status"])
comparison_year_report = comparison_year_report.reindex(columns=COMPARISON_STATUS_ORDER, fill_value=0).reset_index()
comparison_year_report.columns = [
    "Year", "Identity-supported", "Within 500 m", "0.5–5 km", "Over 5 km",
    "No same-day record", "Ecotype conflict", "Not assessable",
]
high_priority_statuses = ["Identity-supported overlap candidate", "Very close same-day candidate"]
high_priority_comparison = comparison.loc[comparison["comparison_status"].isin(high_priority_statuses)].copy()
comparison_source_report = (
    high_priority_comparison["existing_preferred_source"].value_counts()
    .rename_axis("Existing preferred source").rename("CWR candidate matches").reset_index()
)
identity_supported_count = int(comparison["comparison_status"].eq("Identity-supported overlap candidate").sum())
very_close_count = int(comparison["comparison_status"].eq("Very close same-day candidate").sum())
nearby_count = int(comparison["comparison_status"].eq("Nearby same-day candidate").sum())
high_priority_count = identity_supported_count + very_close_count
high_priority_unique_observations = int(high_priority_comparison["existing_observation_id"].nunique())
no_close_evidence_count = len(comparison) - high_priority_count - nearby_count

assert len(comparison) == len(encounters)
assert comparison["source_record_key"].is_unique
assert int(comparison_status_report["CWR encounters"].sum()) == len(encounters)
display(comparison_status_report)
display(comparison_year_report)
print({
    "reference_snapshot_id": comparison_snapshot_id,
    "reference_observations": len(orcacast_observations_all),
    "cwr_reference_rows_excluded": cwr_reference_rows_excluded,
    "identity_supported_candidates": identity_supported_count,
    "very_close_candidates": very_close_count,
    "nearby_candidates": nearby_count,
    "no_close_or_evaluable_evidence": no_close_evidence_count,
})


## HTML report

The report is organized around the configured internal source: extraction architecture, overlap with a CWR-excluded reference snapshot, the relationship to SalishSea.io, permission constraints, and then supporting maps, charts, QC, and provenance. The reusable HTML template lives beside this notebook in `build_cwr_potential_source_report.py`. Plotly is embedded; Folium loads Leaflet and map tiles from their normal internet sources.

In [ ]:
def dataframe_html(frame: pd.DataFrame, index: bool = False) -> str:
    return frame.to_html(index=index, border=0, classes="data-table", escape=True)


report_generated_at_utc = datetime.now(timezone.utc).isoformat(timespec="seconds").replace("+00:00", "Z")
year_report = year_ecotype_counts.reset_index().rename(columns={"source_year": "Source year"})
ecotype_report = ecotype_counts.rename(columns={"ecotype_display": "Ecotype", "encounters": "Encounters"})

coverage_report = (
    encounters.assign(mapped=encounters[["map_lat", "map_lon"]].notna().all(axis=1))
    .groupby(["source_year", "source_system"], as_index=False)
    .agg(Encounters=("source_record_key", "count"), Mapped=("mapped", "sum"))
    .rename(columns={"source_year": "Source year", "source_system": "Source system"})
)
coverage_report["Mapped"] = coverage_report["Mapped"].astype(int)
coverage_report["Map coverage"] = coverage_report.apply(
    lambda row: f"{100 * row['Mapped'] / row['Encounters']:.1f}%", axis=1
)

archive_inventory_rows = []
for year in ARCHIVE_YEARS:
    year_archive = archive.loc[archive["source_year"].eq(year)]
    index_info = archive_cache["index_results"][str(year)]
    archive_inventory_rows.append({
        "Year": year,
        "Index entries": index_info["entry_count"],
        "Canonical encounters": len(year_archive),
        "Standard": int(year_archive["record_series"].eq("encounter").sum()),
        "UAV": int(year_archive["record_series"].eq("uav_encounter").sum()),
        "Mapped": int(year_archive[["map_lat", "map_lon"]].notna().all(axis=1).sum()),
    })
archive_inventory_report = pd.DataFrame(archive_inventory_rows)

qc_counter = Counter(flag for flags in encounters["qc_flags"] for flag in flags)
qc_summary = pd.DataFrame(
    [{"QC flag": flag, "Encounter records": count} for flag, count in qc_counter.most_common()]
)
qc_rows = encounters.loc[
    encounters["qc_flags"].map(bool),
    ["source_year", "record_series", "encounter_number", "source_record_name", "qc_flags"],
].copy()
qc_rows["qc_flags"] = qc_rows["qc_flags"].map(lambda values: ", ".join(values))

time_series_html = time_series_fig.to_html(
    full_html=False,
    include_plotlyjs=True,
    config={"displaylogo": False, "responsive": True},
)
map_document = encounter_map.get_root().render()
map_srcdoc = html.escape(map_document, quote=True)

metric_cards = "".join(
    f'<article class="metric"><span>{html.escape(str(label))}</span><strong>{html.escape(str(value))}</strong></article>'
    for label, value in [
        ("Canonical encounters", f"{len(encounters):,}"),
        ("Archive encounters", f"{len(archive):,}"),
        ("Atlist encounters", f"{len(atlist):,}"),
        ("Mapped encounters", f"{len(mapped):,}"),
        ("Source years", "2017–2026"),
        ("Report generated (UTC)", report_generated_at_utc),
    ]
)
comparison_metric_cards = "".join(
    f'<article class="metric"><span>{html.escape(str(label))}</span><strong>{html.escape(str(value))}</strong></article>'
    for label, value in [
        ("Identity-supported", f"{identity_supported_count:,}"),
        ("Same day + within 500 m", f"{very_close_count:,}"),
        ("Same day + 0.5–5 km", f"{nearby_count:,}"),
        ("No close/evaluable evidence", f"{no_close_evidence_count:,}"),
    ]
)

report_html = f'''<!doctype html>
<html lang="en">
<head>
<meta charset="utf-8">
<meta name="viewport" content="width=device-width, initial-scale=1">
<title>CWR 2017–2026 all-archives encounter report</title>
<style>
:root {{ --ink:#102A43; --muted:#52667A; --line:#D9E2EC; --panel:#F6F9FC; --accent:#0072B2; }}
* {{ box-sizing:border-box; }}
body {{ margin:0; color:var(--ink); background:#EDF3F8; font:15px/1.55 Inter, ui-sans-serif, system-ui, -apple-system, BlinkMacSystemFont, "Segoe UI", sans-serif; }}
main {{ width:min(1220px, calc(100% - 32px)); margin:32px auto 64px; }}
header, section {{ background:white; border:1px solid var(--line); border-radius:14px; box-shadow:0 8px 24px rgba(16,42,67,.06); }}
header {{ padding:32px; border-top:5px solid var(--accent); }}
section {{ margin-top:20px; padding:26px; }}
h1 {{ margin:0 0 8px; font-size:clamp(26px,4vw,42px); line-height:1.12; letter-spacing:-.02em; }}
h2 {{ margin:0 0 14px; font-size:23px; }}
h3 {{ margin:24px 0 10px; font-size:17px; }}
p {{ max-width:88ch; }}
.lede {{ color:var(--muted); font-size:17px; margin:0; }}
.metrics {{ display:grid; grid-template-columns:repeat(auto-fit,minmax(180px,1fr)); gap:12px; margin-top:24px; }}
.metric {{ padding:15px 16px; background:var(--panel); border:1px solid var(--line); border-radius:10px; }}
.metric span {{ display:block; color:var(--muted); font-size:12px; font-weight:700; letter-spacing:.04em; text-transform:uppercase; }}
.metric strong {{ display:block; margin-top:5px; font-size:20px; overflow-wrap:anywhere; }}
.two-col {{ display:grid; grid-template-columns:repeat(2,minmax(0,1fr)); gap:20px; align-items:start; }}
.scroll {{ overflow-x:auto; }}
.data-table {{ width:100%; border-collapse:collapse; font-size:14px; }}
.data-table th, .data-table td {{ padding:9px 10px; text-align:left; border-bottom:1px solid var(--line); vertical-align:top; }}
.data-table th {{ background:var(--panel); font-weight:700; white-space:nowrap; }}
.chart {{ min-height:500px; }}
.map-frame {{ width:100%; min-height:700px; border:1px solid var(--line); border-radius:10px; background:#E9F1F7; }}
.callout {{ padding:14px 16px; border-left:4px solid var(--accent); background:#EDF7FC; border-radius:6px; color:#234E67; }}
.warning {{ border-left-color:#B45309; background:#FFF7ED; color:#7C2D12; }}
a {{ color:#005A8D; }}
code {{ background:#EEF2F6; padding:.1em .3em; border-radius:4px; }}
footer {{ color:var(--muted); margin-top:20px; font-size:13px; text-align:center; }}
@media (max-width:800px) {{ .two-col {{ grid-template-columns:1fr; }} header, section {{ padding:20px; }} .map-frame {{ min-height:540px; }} }}
</style>
</head>
<body>
<main>
<header>
  <h1>CWR 2017–2026 all-archives encounter report</h1>
  <p class="lede">Every encounter year exposed by the CWR archive index, combined with the existing 2024–2026 Atlist export.</p>
  <div class="metrics">{metric_cards}</div>
</header>

<section>
  <h2>Scope and interpretation</h2>
  <p class="callout">The linked archive exposes encounter years 2017–2023. Although the site notes that Orca Survey began in 1976, it does not expose earlier encounter-year archives in its navigation; those years are unavailable here rather than counted as zero.</p>
  <p>One row is one canonical encounter within a source year and record series. Three multi-sequence archive encounters are aggregated from their component pages. UAV encounter series remain distinct from standard encounters. Counts are reporting records—not independent sightings, abundance estimates, or effort-corrected occurrence rates.</p>
</section>

<section>
  <h2>At a glance</h2>
  <div class="two-col">
    <div class="scroll"><h3>Total by ecotype</h3>{dataframe_html(ecotype_report)}</div>
    <div class="scroll"><h3>Source-year totals</h3>{dataframe_html(year_report)}</div>
  </div>
</section>

<section>
  <h2>Monthly encounter records</h2>
  <p>Zeroes are filled from January 2017 through {latest_in_year_month.strftime('%B %Y')}, the latest in-year 2026 encounter month in the Atlist input. Later months are not represented as zero.</p>
  <div class="chart">{time_series_html}</div>
</section>

<section>
  <h2>Encounter map</h2>
  <p>Archive records use source-reported start coordinates where available; end coordinates are used only as an explicit fallback. Unsigned archive longitudes are interpreted as west within the CWR Pacific Northwest study domain. Parsed candidates outside a broad 45–52°N, 120–130°W plausibility envelope are retained in provenance fields and QC flags but withheld from the map. Atlist records retain their map-marker coordinates.</p>
  <iframe class="map-frame" title="CWR 2017–2026 encounter map" srcdoc="{map_srcdoc}"></iframe>
  <h3>Map coverage by source year</h3>
  <div class="scroll">{dataframe_html(coverage_report)}</div>
</section>

<section>
  <h2>Archive inventory and quality controls</h2>
  <div class="scroll">{dataframe_html(archive_inventory_report)}</div>
  <h3>QC summary</h3>
  <div class="scroll">{dataframe_html(qc_summary) if len(qc_summary) else '<p>None.</p>'}</div>
  <details><summary>QC-flagged records ({len(qc_rows):,})</summary><div class="scroll">{dataframe_html(qc_rows)}</div></details>
  <p class="callout warning">One 2020 archive page URL is reused by encounter entries #1 and #2. The index identity is retained for both, but page-level fields are withheld from the mismatched entry. One 2017 <code>CA/U</code> label remains unknown rather than being assigned an ecotype.</p>
</section>

<section>
  <h2>Data and provenance</h2>
  <p>Archive pull: <strong>{html.escape(archive_retrieved_at_utc)}</strong>. Existing Atlist pull(s): <strong>{html.escape(', '.join(atlist_pull_times))}</strong>. Report generated: <strong>{html.escape(report_generated_at_utc)}</strong>.</p>
  <p>Archive rows retain every component page URL and checksum, the source index checksum, original structured fields, coordinate source text, parse method, and QC flags. Full archive narratives and images are not copied into this dataset.</p>
  <ul>
    <li><a href="{ARCHIVE_INDEX_URL}">CWR Archive Encounters index</a></li>
    <li><a href="https://www.whaleresearch.com/encounters2024">CWR 2024 encounters</a></li>
    <li><a href="https://www.whaleresearch.com/encounters">CWR 2025 encounters</a></li>
    <li><a href="https://www.whaleresearch.com/encounters-map-2026">CWR 2026 encounters</a></li>
  </ul>
  <p>Blank fields mean the source did not state or expose a value. Time zones are not inferred. Confirm permission and redistribution terms before publishing source details.</p>
</section>

<section id="existing-source-comparison">
  <h2>Are these encounters already in the existing sightings dataset?</h2>
  <p class="callout"><strong>Bottom line:</strong> {identity_supported_count:,} CWR encounters have exact same-day whale or Bigg's social-group evidence in the existing canonical data, and {very_close_count:,} additional encounters have a non-conflicting same-day observation within 500 m. Together, these {high_priority_count:,} high-priority candidates point to {high_priority_unique_observations:,} unique existing canonical observations. Another {nearby_count:,} encounters have a nearby same-day candidate within 0.5–5 km. The remaining {no_close_evidence_count:,} have no close or evaluable overlap evidence in this screen; that is not proof that they are new.</p>
  <div class="metrics">{comparison_metric_cards}</div>
  <div class="two-col">
    <div class="scroll"><h3>Screening result</h3>{dataframe_html(comparison_status_report)}</div>
    <div class="scroll"><h3>Preferred source for high-priority candidates</h3>{dataframe_html(comparison_source_report)}</div>
  </div>
  <h3>Results by CWR source year</h3>
  <div class="scroll">{dataframe_html(comparison_year_report)}</div>
  <h3>Comparison method</h3>
  <p>The reference is normalized sightings snapshot <code>{html.escape(comparison_snapshot_id)}</code>, built {html.escape(comparison_snapshot_created_at)}, with {len(orcacast_observations_all):,} canonical observations from {html.escape(', '.join(comparison_snapshot_sources))}.</p>
  <ul>
    <li><strong>Identity-supported:</strong> exact calendar date plus at least one shared named whale or Bigg's social-group identifier. Broad J/K/L pod labels and ecotype labels alone are not used as identity evidence.</li>
    <li><strong>Very close:</strong> exact calendar date, no conflicting known ecotype, and the closest existing coordinate is within 500 m.</li>
    <li><strong>Nearby:</strong> the same test at 0.5–5 km. This is possible overlap, not confirmation.</li>
    <li>Canonical <code>UNKNOWN</code> or <code>MIXED</code> ecotypes are treated as non-conflicting; known contradictory ecotypes are excluded.</li>
  </ul>
  <p class="callout warning">CWR is not yet a native configured source in this snapshot, so this is a candidate-overlap screen rather than a pipeline deduplication result. Encounter start coordinates can differ from another report made later along the same whale movement, while same-day proximity can also occur by chance. Ingest CWR with source IDs and run the canonical identity resolver before labeling individual records as duplicates or additive.</p>
</section>
<footer>Generated by <code>03_CWR_2017_2026_ALL_ARCHIVES_REPORT_AND_EXPORT.ipynb</code></footer>
</main>
</body>
</html>
'''

# Keep the decision-oriented report template readable and reusable outside the notebook.
report_builder_path = NOTEBOOK_DIR / "build_cwr_potential_source_report.py"
exec(report_builder_path.read_text(encoding="utf-8"), globals())


## Interpretation notes

- This report is additive and does not replace the existing 2024–2026 notebook, CSV, or HTML report.
- The CWR archive index exposes 2017–2023 encounter years; earlier survey years are unavailable in that index.
- Standard and UAV series remain separate; three multi-sequence encounters are aggregated with page-level lineage retained.
- Archive coordinates come directly from structured source fields and are not geocoded from location text.
- Unsigned archive longitudes use an explicit west-hemisphere interpretation appropriate to the source study area.
- One mismatched shared 2020 page is not allowed to overwrite its index encounter identity.
- The existing-source comparison is a candidate-overlap screen. Exact identifiers, same-day spatial distance, provenance, and ecotype compatibility remain separate evidence fields; no candidate is silently declared a duplicate.
- Missing/unmapped values remain null or `Unknown / not stated`; they are not converted to zero.
- The source and derived cache are snapshots. Set `REFRESH_ARCHIVE_CACHE = True` to pull the live pages again.
